# Agent Network Data Audit and Clean-Up

ADEN AHMED MOHAMED — Cohort 1M DEVIS

Audit and clean the supplied seed-44 synthetic transactions before county and weekly reporting.

## Task 10: Load and audit the raw data

Audit raw rows before deduplication; counts overlap and are not additive. Default pandas CSV parsing treats both blank amounts and N/A as missing; a second literal-text read distinguishes those source representations. Statistical outliers are review candidates, not proven errors. Six dimensions are completeness, uniqueness, consistency, validity, accuracy and timeliness.

In [1]:
from pathlib import Path
import runpy
import pandas as pd
import numpy as np
from cleaning_pipeline import clean_agent_data

if not all(Path(name).exists() for name in ["agents.csv", "agent_transactions.csv"]):
    runpy.run_path("generate_data.py")
txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")
literal_txns = pd.read_csv("agent_transactions.csv", keep_default_na=False)
raw_backup = txns.copy(deep=True)
amount_numeric = pd.to_numeric(txns["amount"], errors="coerce")
canonical_type = txns.txn_type.astype("string").str.strip().str.title()
normalized_dates = pd.to_datetime(txns.txn_date.str.replace("/", "-", regex=False), format="%Y-%m-%d", errors="coerce")
valid_amounts = amount_numeric.where(np.isfinite(amount_numeric) & amount_numeric.gt(0))
q1, q3 = valid_amounts.quantile([0.25, 0.75])
raw_outlier_mask = valid_amounts.lt(q1 - 1.5 * (q3-q1)) | valid_amounts.gt(q3 + 1.5 * (q3-q1))
duplicate_records = txns.loc[txns.txn_id.duplicated(keep=False)]
agent_conflict_groups = duplicate_records.groupby("txn_id").agent_id.nunique().gt(1).sum()
audit = pd.DataFrame([
    ["Repeated transaction IDs (extra rows)", "Uniqueness", "txn_id", txns.txn_id.duplicated().sum()],
    ["Exact duplicate records (extra rows)", "Uniqueness", "all source fields", txns.duplicated().sum()],
    ["Rows in conflicting-agent duplicate groups", "Consistency", "txn_id / agent_id", duplicate_records.txn_id.isin(duplicate_records.groupby("txn_id").agent_id.nunique().loc[lambda x: x.gt(1)].index).sum()],
    ["Missing amounts, including default-parsed N/A", "Completeness", "amount", txns.amount.isna().sum()],
    ["Literal blank amounts (subset of missing)", "Completeness", "amount", literal_txns.amount.eq("").sum()],
    ["Literal N/A tokens (subset of missing)", "Validity", "amount", literal_txns.amount.eq("N/A").sum()],
    ["Noncanonical transaction-type labels", "Consistency", "txn_type", txns.txn_type.ne(canonical_type).sum()],
    ["Non-ISO slash-separated dates", "Consistency", "txn_date", txns.txn_date.str.contains("/", na=False).sum()],
    ["Dates unparseable after separator repair", "Validity", "txn_date", normalized_dates.isna().sum()],
    ["Unknown agent references", "Validity", "agent_id", (~txns.agent_id.isin(agents.agent_id)).sum()],
    ["IQR outliers requiring verification", "Accuracy", "amount", raw_outlier_mask.sum()],
    ["KES 250,000 amounts requiring verification", "Accuracy", "amount", amount_numeric.eq(250000).sum()],
    ["Missing extract timestamp; freshness unverifiable", "Timeliness", "extract_timestamp (not provided)", len(txns)],
], columns=["issue", "dimension", "column", "rows affected"])
audit["rows affected"] = audit["rows affected"].astype(int)
print("Raw transactions:", txns.shape, "Agent register:", agents.shape)
display(audit)
print("Conflicting-agent duplicate groups:", agent_conflict_groups)
display(duplicate_records.sort_values("txn_id"))
print("Missing register keys:", agents.agent_id.isna().sum())
print("Duplicate register keys:", agents.agent_id.duplicated().sum())
print("Observed transaction dates:", normalized_dates.min(), "to", normalized_dates.max())

Raw transactions: (612, 5) Agent register: (20, 3)


,issue,dimension,column,rows affected
0,Repeated transaction IDs (extra rows),Uniqueness,txn_id,12
1,Exact duplicate records (extra rows),Uniqueness,all source fields,11
2,Rows in conflicting-agent duplicate groups,Consistency,txn_id / agent_id,2
3,"Missing amounts, including default-parsed N/A",Completeness,amount,22
4,Literal blank amounts (subset of missing),Completeness,amount,15
5,Literal N/A tokens (subset of missing),Validity,amount,7
6,Noncanonical transaction-type labels,Consistency,txn_type,384
7,Non-ISO slash-separated dates,Consistency,txn_date,5
8,Dates unparseable after separator repair,Validity,txn_date,0
9,Unknown agent references,Validity,agent_id,8


Conflicting-agent duplicate groups: 1


,txn_id,agent_id,txn_type,amount,txn_date
23,TX00024,AG017,DEPOSIT,2798.0,2026-04-28
605,TX00024,AG017,DEPOSIT,2798.0,2026-04-28
66,TX00067,AG002,deposit,NaN,2026-04-07
608,TX00067,AG002,deposit,NaN,2026-04-07
165,TX00166,AG010,deposit,4146.0,2026-04-18
609,TX00166,AG010,deposit,4146.0,2026-04-18
201,TX00202,AG010,Deposit,2243.0,2026-03-22
604,TX00202,AG010,Deposit,2243.0,2026-03-22
611,TX00330,AG016,withdrawal,1219.0,2026-04-07
329,TX00330,AG016,withdrawal,1219.0,2026-04-07


Missing register keys: 0
Duplicate register keys: 0
Observed transaction dates: 2026-03-01 00:00:00 to 2026-04-29 00:00:00


## Tasks 11–14 and 18: Run the reusable cleaning pipeline

The notebook imports and calls clean_agent_data(txns, agents) from cleaning_pipeline.py. It deduplicates by txn_id, resolves the single agent-only conflict using the register, standardises transaction labels, converts amounts and dates, and flags missing/invalid amounts before median imputation. Median imputation preserves transaction counts and is less affected by the right-skewed amounts and the five KES 250,000 values than mean imputation; imputed values remain estimates and should not be used for ledger settlement.

In [2]:
clean_df, log = clean_agent_data(txns, agents)
cleaning_log = pd.DataFrame(log)
display(cleaning_log)
display(clean_df.dtypes.rename("dtype").to_frame())
print("Before:", len(txns), "After:", len(clean_df))
print("Canonical transaction types:", sorted(clean_df.txn_type.unique()))
print("Imputed amounts:", clean_df.amount_imputed.sum())
print("Imputation value:", clean_df.loc[clean_df.amount_imputed, "amount"].unique())
pd.testing.assert_frame_equal(txns, raw_backup)
assert len(txns) - len(clean_df) == txns.txn_id.duplicated().sum()
assert clean_df.txn_id.is_unique
assert clean_df.amount.notna().all()
assert clean_df.txn_type.isin(["Deposit", "Withdrawal"]).all()
assert pd.api.types.is_datetime64_any_dtype(clean_df.txn_date)
assert pd.api.types.is_numeric_dtype(clean_df.amount)

,rule,rows_affected,detail
0,Input rows,612,Raw transaction extract
1,Duplicate IDs with conflicting agents,1,Prefer registered ID only when all other field...
2,Duplicate transactions removed,12,One record per txn_id
3,Transaction labels standardised,376,Trimmed whitespace and normalised case
4,Amounts imputed,20,"Median of observed valid amounts = KES 1,776.50"
5,IQR outliers retained,40,"Observed-value fences [-1927.50, 6084.50]; no ..."
6,"KES 250,000 records retained",5,Check source receipts/ledger and authorisation...
7,Date separators normalised,5,YYYY/MM/DD converted to datetime
8,Transactions with unknown agents,7,Preserved under county Unknown; no guessed county
9,Transactions before agent onboarding,0,Temporal plausibility check; NaT for unknown a...


,dtype
txn_id,object
agent_id,object
txn_type,string[python]
amount,float64
txn_date,datetime64[ns]
amount_imputed,bool
is_outlier,bool
county,object
onboarded,datetime64[ns]
unknown_agent,bool


Before: 612 After: 600
Canonical transaction types: ['Deposit', 'Withdrawal']
Imputed amounts: 20
Imputation value: [1776.5]


**Answer:** The pipeline removes **12** extra transaction-ID rows (only **11** are exact duplicates after the injected agent-ID change), leaving **600** unique transactions. One repeated ID disagrees only on agent_id; the registered copy is kept and the decision is logged rather than guessed silently. It standardises **376** labels, repairs **5** slash-separated dates, and imputes **20** amounts at the observed valid median of **KES 1,776.50**, flagged by amount_imputed; neither input DataFrame is modified.

## Task 15: Outlier investigation

Apply both 1.5 × IQR fences to observed valid deduplicated amounts before imputation, retain outliers and distinguish an anomaly flag from a confirmed error. Compare totals with and without outliers and show the five KES 250,000 transactions explicitly.

In [3]:
observed = clean_df.loc[~clean_df.amount_imputed, "amount"]
q1, q3 = observed.quantile([0.25, 0.75])
lower_fence, upper_fence = q1 - 1.5*(q3-q1), q3 + 1.5*(q3-q1)
print(f"Q1: {q1:,.2f}; Q3: {q3:,.2f}; IQR fences: [{lower_fence:,.2f}, {upper_fence:,.2f}]")
print("Flagged outliers:", clean_df.is_outlier.sum())
high_value_records = clean_df.loc[clean_df.amount.eq(250000)]
display(high_value_records[["txn_id", "agent_id", "txn_type", "amount", "txn_date", "county", "is_outlier"]])
sensitivity = pd.DataFrame({
    "scenario": ["All retained transactions", "Exclude flagged outliers (sensitivity only)", "Observed amounts only (exclude imputations)"],
    "transactions": [len(clean_df), (~clean_df.is_outlier).sum(), (~clean_df.amount_imputed).sum()],
    "total_KES": [clean_df.amount.sum(), clean_df.loc[~clean_df.is_outlier, "amount"].sum(), clean_df.loc[~clean_df.amount_imputed, "amount"].sum()]
})
display(sensitivity)
print(f"KES 250,000 records' share of total: {high_value_records.amount.sum()/clean_df.amount.sum()*100:.2f}%")

Q1: 1,077.00; Q3: 3,080.00; IQR fences: [-1,927.50, 6,084.50]
Flagged outliers: 40


,txn_id,agent_id,txn_type,amount,txn_date,county,is_outlier
306,TX00307,AG012,Deposit,250000.0,2026-03-02,Kisumu,True
315,TX00316,AG014,Deposit,250000.0,2026-04-01,Nairobi,True
424,TX00425,AG005,Withdrawal,250000.0,2026-03-02,Mombasa,True
463,TX00464,AG009,Withdrawal,250000.0,2026-04-11,Nakuru,True
594,TX00595,AG016,Deposit,250000.0,2026-03-11,Kiambu,True


,scenario,transactions,total_KES
0,All retained transactions,600,2688427.0
1,Exclude flagged outliers (sensitivity only),560,1123268.0
2,Observed amounts only (exclude imputations),580,2652897.0


KES 250,000 records' share of total: 46.50%


**Answer:** There are **40** IQR outliers, including all **5** KES 250,000 records, with an upper fence of **KES 6,084.50**. Reconcile their transaction IDs against the source ledger, receipts and authorisation records; they could be genuine high-value activity or entry errors. Retain them in the reported total with is_outlier=True until confirmed, and show the sensitivity comparison above so management can see their influence. Any source-approved correction should be recorded with original value, corrected value, reviewer and date.

## Task 16: Validate the agent merge

Use a left join with validate="many_to_one" so no valid transaction disappears and no duplicate register key multiplies rows. Unknown references retain agent_id, use county Unknown, and have unknown_agent=True; onboarded remains missing because it is not known.

In [4]:
deduplicated_count = txns.txn_id.nunique()
validation = pd.DataFrame({
    "metric": ["Raw rows", "Unique transaction IDs before merge", "Rows after left merge", "Raw unknown-agent references", "Clean unknown-agent references", "Before-onboarding transactions"],
    "value": [len(txns), deduplicated_count, len(clean_df), (~txns.agent_id.isin(agents.agent_id)).sum(), clean_df.unknown_agent.sum(), clean_df.txn_date.lt(clean_df.onboarded).sum()]
})
display(validation)
display(clean_df.loc[clean_df.unknown_agent])
assert len(clean_df) == deduplicated_count
assert clean_df.county.notna().all()
assert clean_df.loc[clean_df.unknown_agent, "county"].eq("Unknown").all()
print("Inner-join loss avoided:", clean_df.unknown_agent.sum(), "transactions")
print(f"Value preserved under Unknown: KES {clean_df.loc[clean_df.unknown_agent,'amount'].sum():,.2f}")

,metric,value
0,Raw rows,612
1,Unique transaction IDs before merge,600
2,Rows after left merge,600
3,Raw unknown-agent references,8
4,Clean unknown-agent references,7
5,Before-onboarding transactions,0


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county,onboarded,unknown_agent
55,TX00056,AG099,Deposit,8507.0,2026-04-01,False,True,Unknown,NaT,True
68,TX00069,AG099,Withdrawal,1877.0,2026-04-24,False,False,Unknown,NaT,True
77,TX00078,AG099,Deposit,1770.0,2026-04-06,False,False,Unknown,NaT,True
237,TX00238,AG099,Deposit,4944.0,2026-04-06,False,False,Unknown,NaT,True
267,TX00268,AG099,Deposit,3110.0,2026-04-09,False,False,Unknown,NaT,True
444,TX00445,AG099,Deposit,5309.0,2026-04-27,False,False,Unknown,NaT,True
557,TX00558,AG099,Deposit,1639.0,2026-03-22,False,False,Unknown,NaT,True


Inner-join loss avoided: 7 transactions
Value preserved under Unknown: KES 27,156.00


**Answer:** The raw extract has **8** AG099 references; after resolving the agent-only duplicate conflict and deduplicating, **7** unique transactions remain unmatched. The left merge preserves all **600** rows. These seven remain in reporting under Unknown and should be reconciled by the register owner; they are not assigned to an invented county. No matched transaction predates its agent’s onboarding date.

## Task 17: County, agent and weekly summaries

Split county value/count/average by transaction type, include Unknown to reconcile every transaction, and aggregate weekly values using Monday–Sunday weeks labelled by Monday start date. Also show leading agents to address the manager’s agent-level question. Totals include retained outliers and median-imputed estimates.

In [5]:
county_summary = clean_df.groupby(["county", "txn_type"], as_index=False).agg(
    total_value=("amount", "sum"), transactions=("txn_id", "size"),
    average_value=("amount", "mean"), imputed_amounts=("amount_imputed", "sum"),
    outliers=("is_outlier", "sum")
).sort_values(["county", "txn_type"])
display(county_summary.round(2))
weekly_data = clean_df.assign(week_start=clean_df.txn_date.dt.to_period("W-SUN").dt.start_time)
weekly_pivot = weekly_data.pivot_table(index="week_start", columns="county", values="amount", aggfunc="sum", fill_value=0)
weekly_pivot.index = weekly_pivot.index.strftime("%Y-%m-%d")
display(weekly_pivot)
agent_summary = clean_df.groupby(["agent_id", "county", "txn_type"], as_index=False).agg(
    total_value=("amount", "sum"), transactions=("txn_id", "size")
).sort_values(["txn_type", "total_value"], ascending=[True, False])
display(agent_summary.groupby("txn_type", sort=False).head(3))
assert county_summary.transactions.sum() == len(clean_df)
assert np.isclose(county_summary.total_value.sum(), clean_df.amount.sum())
assert np.isclose(weekly_pivot.to_numpy().sum(), clean_df.amount.sum())
print(f"County and weekly totals reconcile: KES {clean_df.amount.sum():,.2f}")
Path("outputs").mkdir(exist_ok=True)
clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False, date_format="%Y-%m-%d")
exported = pd.read_csv("outputs/agent_transactions_clean.csv")
assert exported.txn_id.is_unique
assert exported.txn_date.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()
print("Exported 600 unique transactions with ISO dates, county and audit flags.")

,county,txn_type,total_value,transactions,average_value,imputed_amounts,outliers
0,Kiambu,Deposit,334737.0,43,7784.58,0,1
1,Kiambu,Withdrawal,77050.5,27,2853.72,1,4
2,Kisumu,Deposit,436420.5,74,5897.57,3,7
3,Kisumu,Withdrawal,76374.5,37,2064.18,1,2
4,Mombasa,Deposit,148986.0,72,2069.25,4,3
5,Mombasa,Withdrawal,368592.0,61,6042.49,0,2
6,Nairobi,Deposit,543590.0,118,4606.69,4,8
7,Nairobi,Withdrawal,162036.0,62,2613.48,4,5
8,Nakuru,Deposit,184890.0,60,3081.50,2,5
9,Nakuru,Withdrawal,328594.5,39,8425.50,1,2


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown
week_start,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2743.5,0.0
2026-03-02,18317.0,287933.0,285669.5,68967.0,56574.0,0.0
2026-03-09,267771.5,23186.0,30477.5,52860.5,28730.0,0.0
2026-03-16,22054.0,28125.0,25144.5,43107.5,39795.0,1639.0
2026-03-23,11504.0,27077.0,17050.0,41373.5,18998.5,0.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06,15030.0,15341.0,47092.5,56786.5,276609.0,9824.0
2026-04-13,12265.0,29416.0,26730.0,56432.5,26152.5,0.0
2026-04-20,24507.0,37439.0,16687.0,65187.0,30862.0,1877.0


,agent_id,county,txn_type,total_value,transactions
26,AG014,Nairobi,Deposit,305829.0,22
30,AG016,Kiambu,Deposit,293758.0,20
22,AG012,Kisumu,Deposit,282896.0,16
17,AG009,Nakuru,Withdrawal,280184.5,14
9,AG005,Mombasa,Withdrawal,274721.0,15
31,AG016,Kiambu,Withdrawal,41416.0,13


County and weekly totals reconcile: KES 2,688,427.00
Exported 600 unique transactions with ISO dates, county and audit flags.


**Answer:** Nairobi leads reported county value at **KES 705,626.00**, while deposits total **KES 1,673,902.50** and withdrawals **KES 1,014,524.50** across the network. The **Unknown** county retains **KES 27,156.00** and is included in both county and weekly totals. These values are transaction throughput, not provider revenue or profit, and the first/last reporting weeks can be partial.

## Data Quality Report

**Scope:** Seed-44 synthetic extract, **612 raw rows → 600 unique transactions**, with **20 agents**; report values include imputed estimates and retained outliers.

### 1. Completeness
**20** missing/invalid amounts were flagged before imputation at the observed median, **KES 1,776.50**; estimated amounts contribute **KES 35,530.00** and must be replaced with verified source amounts before financial reconciliation.

### 2. Uniqueness
Removed **12** extra txn_id rows; only **11** were exact full-row duplicates because one duplicate group had conflicting agent IDs, resolved by preferring the registered ID when all financial/date/type fields matched; no distinct transaction was dropped.

### 3. Consistency
Standardised **376** labels to Deposit or Withdrawal and repaired **5** slash-separated dates; export uses YYYY-MM-DD, consistent numeric amounts and boolean review flags.

### 4. Validity
After a many-to-one left merge, **600** rows remain and **7** unknown-agent transactions worth **KES 27,156.00** are preserved under Unknown; the register owner should resolve AG099 rather than silently exclude or invent its county.

### 5. Accuracy
Flagged **40** observed IQR outliers, including **5** KES 250,000 entries totalling **KES 1,250,000** (46.50% of reported throughput); retain them pending receipt/ledger checks, show sensitivity results, and record approved corrections because statistical flags cannot prove inaccuracy.

### 6. Timeliness
Dates cover **01 March 2026–29 April 2026** and no matched transaction predates onboarding, but extract/arrival timestamps and an SLA are absent, so ingestion delay and freshness cannot be assessed; add these metadata to future deliveries.

### Privacy and ownership
Customer phone numbers would be direct personal identifiers; linked txn_id, agent_id, amount, txn_date and county could also be personal data when they identify or can be linked to a person. For the Kenya Data Protection Act 2019, define a lawful purpose, minimise collected fields and retention, provide appropriate notice, and protect access with least privilege and encryption in transit/at rest. Remove phone numbers from analysis outputs and public GitHub, use controlled tokens with separately secured mappings, and restrict detailed real transactions because pseudonymisation alone does not make them anonymous. This repository contains generated synthetic records only; source-system owners should verify amounts and the agent register before operational use.

Sources: [Kenya Data Protection Act 2019, sections 2, 25 and 41](https://odpc.go.ke/wp-content/uploads/2024/02/TheDataProtectionAct__No24of2019.pdf); [ODPC personal-data guidance](https://www.odpc.go.ke/faqs/).